# Mini Trainer — Train on Extracted Features

This notebook demonstrates how to:

1. **Extract features** from a frozen foundation model (MACE or any `FeatureExtractorBackbone`)
2. **Cache** those features as a lightweight `TensorDataset`
3. **Train** your own downstream model using the standalone `MiniTrainer`

The `MiniTrainer` is completely decoupled from the Lightning / Hydra training
pipeline. It operates on raw PyTorch primitives: a model, a loss, an optimiser,
and a data loader.

> **Why?** Foundation models like MACE produce rich equivariant representations.
> Extracting and caching them once lets you iterate on downstream heads *fast* —
> no need to re-run the expensive backbone forward pass each epoch.

In [ ]:
from __future__ import annotations

import typing

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Build Dummy Atomic Structures

We create a small set of `AtomicGraph` objects from inline ASE `Atoms`.
In a real workflow you would load your dataset via the GOAL data pipeline
(e.g. `GOALDataModule` with an XYZ/LMDB/HDF5 config).

In [ ]:
from ase import Atoms
from torch_geometric.data import Batch

from goal.ml.data.graph import AtomicGraph

# --- Water molecule ---
water: Atoms = Atoms(
    "OH2",
    positions=[
        [0.000, 0.000, 0.119],
        [0.000, 0.763, -0.477],
        [0.000, -0.763, -0.477],
    ],
)

# --- Ethanol molecule ---
ethanol: Atoms = Atoms(
    "C2H6O",
    positions=[
        [-0.047, 0.537, 0.000],
        [-1.263, -0.360, 0.000],
        [1.177, -0.181, 0.000],
        [-0.047, 1.158, 0.890],
        [-0.047, 1.158, -0.890],
        [-1.263, -0.981, 0.890],
        [-1.263, -0.981, -0.890],
        [-2.170, 0.244, 0.000],
        [1.177, -0.790, 0.882],
    ],
)

cutoff: float = 5.0
# Assign dummy energies (kcal/mol) and forces
structures: typing.List[typing.Tuple[Atoms, float]] = [
    (water, -76.4),
    (ethanol, -154.8),
]

graphs: typing.List[AtomicGraph] = []
for atoms, energy in structures:
    g: AtomicGraph = AtomicGraph.from_ase(
        atoms,
        cutoff=cutoff,
        energy=energy,
    )
    graphs.append(g)

batched: Batch = Batch.from_data_list(graphs)
print(f"Batched graph: {batched.num_graphs} graphs, {batched.num_nodes} total atoms")

## 2. Load a Foundation Model and Extract Features

We load a MACE backbone, freeze it, and extract multi-scale features.
Then we keep only the L=0 (scalar / invariant) channels — these are
safe to pass to any standard MLP.

> **Note**: If you don't have `mace-torch` installed, skip to *Section 2b*
> below which uses a synthetic feature matrix for demonstration purposes.

In [ ]:
# ===== Section 2a: Real MACE extraction (requires mace-torch) =====
# Uncomment this block if mace-torch is installed.

# from goal.ml.utils.extraction import (
#     FrozenBackbone, MultiScaleBackbone, extract_scalars, pool_nodes,
# )
# from goal.ml.adapters.mace import MACEAdapter
# from goal.ml.data.graph import NodeFeatures
#
# adapter = MACEAdapter.from_pretrained("small")
# frozen = FrozenBackbone(MultiScaleBackbone(adapter))
#
# # Extract scalar features and pool to per-graph representation
# with torch.no_grad():
#     feats = frozen.forward(batched)
#     scalars = extract_scalars(feats)  # (N_total, scalar_dim)
#     graph_features = pool_nodes(
#         NodeFeatures(node_feats=scalars, irreps=f"{scalars.shape[-1]}x0e"),
#         batched.batch, batched.num_graphs, mode="sum",
#     )  # (B, scalar_dim)
#
# feature_dim: int = graph_features.shape[-1]
# target_energies = batched.energy.float()
# print(f"Feature dim: {feature_dim}, targets shape: {target_energies.shape}")

# ===== Section 2b: Synthetic features for demo =====
# This creates a toy regression dataset so the notebook runs without MACE.

N_SAMPLES: int = 200
FEATURE_DIM: int = 128

torch.manual_seed(42)
# Random features pretending to be extracted from a foundation model
X: torch.Tensor = torch.randn(N_SAMPLES, FEATURE_DIM)
# Targets: a linear combination with noise (simulating energies)
true_weights: torch.Tensor = torch.randn(FEATURE_DIM)
y: torch.Tensor = X @ true_weights + 0.1 * torch.randn(N_SAMPLES)

# Normalise targets to realistic energy range
y = y - y.mean()
y = y / y.std() * 10.0  # ≈ 10 kcal/mol spread

feature_dim: int = FEATURE_DIM
print(f"Synthetic dataset: {N_SAMPLES} samples, feature_dim={feature_dim}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}]")

## 3. Prepare DataLoaders

Wrap the extracted features (or synthetic ones) into standard PyTorch
`TensorDataset` + `DataLoader`. This is where caching extracted features
pays off — the data is tiny tensors in memory, no backbone forward pass.

In [ ]:
# Train / val / test split (70 / 15 / 15)
n_train: int = int(0.7 * N_SAMPLES)
n_val: int = int(0.15 * N_SAMPLES)

# Shuffle indices
perm: torch.Tensor = torch.randperm(N_SAMPLES)
train_idx: torch.Tensor = perm[:n_train]
val_idx: torch.Tensor = perm[n_train : n_train + n_val]
test_idx: torch.Tensor = perm[n_train + n_val :]

train_ds: TensorDataset = TensorDataset(X[train_idx], y[train_idx])
val_ds: TensorDataset = TensorDataset(X[val_idx], y[val_idx])
test_ds: TensorDataset = TensorDataset(X[test_idx], y[test_idx])

BATCH_SIZE: int = 32
train_loader: DataLoader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader: DataLoader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader: DataLoader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 4. Define a Downstream Head

A simple MLP head that takes the scalar features and predicts a
single scalar output (energy). In a real setup you might use one
of GOAL's registered heads, or design something richer.

In [ ]:
class MLPHead(nn.Module):
    """Simple MLP for scalar regression on extracted features."""

    def __init__(self, in_dim: int, hidden_dim: int = 128, out_dim: int = 1) -> None:
        super().__init__()
        self.net: nn.Sequential = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


head: MLPHead = MLPHead(in_dim=feature_dim, hidden_dim=128)
n_params: int = sum(p.numel() for p in head.parameters())
print(f"Head parameters: {n_params:,}")

## 5. Train with MiniTrainer

The `MiniTrainer` provides:
- Per-epoch train/val loss logging
- Optional early stopping
- Best-model checkpointing (in-memory)
- Learning rate scheduling
- Gradient clipping
- A `TrainingHistory` object for plotting

In [ ]:
from goal.ml.utils.mini_trainer import MiniTrainer

optimizer: torch.optim.AdamW = torch.optim.AdamW(
    head.parameters(), lr=1e-3, weight_decay=1e-5,
)
scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5, min_lr=1e-6,
    )
)

trainer: MiniTrainer = MiniTrainer(
    model=head,
    loss_fn=nn.MSELoss(),
    optimizer=optimizer,
    scheduler=scheduler,
    device="auto",
    grad_clip=10.0,
)

print(f"Training on: {trainer.device}")

In [ ]:
from goal.ml.utils.mini_trainer import TrainingHistory

history: TrainingHistory = trainer.fit(
    train_loader,
    val_loader=val_loader,
    epochs=100,
    early_stopping_patience=20,
    checkpoint_best=True,
    verbose=True,
)

## 6. Visualise Results

In [ ]:
history.plot()

In [ ]:
print(f"Best val loss:  {history.best_val_loss:.6f}")
print(f"Best epoch:     {history.best_epoch + 1}")
print(f"Total time:     {sum(history.epoch_times):.1f}s")

## 7. Evaluate on Test Set

Restore the best checkpoint and run predictions on the held-out test set.

In [ ]:
# Load the best checkpoint
trainer.load_best()

preds: torch.Tensor
targets: torch.Tensor
preds, targets = trainer.predict(test_loader)

mae: float = (preds - targets).abs().mean().item()
rmse: float = ((preds - targets) ** 2).mean().sqrt().item()
print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(targets.numpy(), preds.numpy(), alpha=0.6, s=20)
lims: typing.List[float] = [
    min(targets.min().item(), preds.min().item()) - 1,
    max(targets.max().item(), preds.max().item()) + 1,
]
ax.plot(lims, lims, "k--", alpha=0.4, label="ideal")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Target")
ax.set_ylabel("Predicted")
ax.set_title(f"Test — MAE={mae:.4f}, RMSE={rmse:.4f}")
ax.set_aspect("equal")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Using MiniTrainer with a Custom Step Function

For more complex scenarios (e.g. training on `AtomicGraph` batches
with a full backbone + head and `CompositeLoss`), pass a custom
`step_fn` to `MiniTrainer`. The built-in `graph_step` handles
this pattern:

In [ ]:
from goal.ml.utils.mini_trainer import graph_step

# Example (not executed — requires a full backbone + head + datamodule):
#
# from goal.ml.registry import BACKBONE_REGISTRY, HEAD_REGISTRY
# from goal.ml.training.loss import (
#     CompositeLoss, WeightedLoss, EnergyLoss, ForcesLoss,
# )
#
# backbone = BACKBONE_REGISTRY.build("hyperspec", num_elements=120, ...)
# head = HEAD_REGISTRY.build("energy_forces", irreps_in=str(backbone.irreps_out), ...)
#
# # Compose your model as a simple nn.Module
# class MyModel(nn.Module):
#     def __init__(self, backbone, head):
#         super().__init__()
#         self.backbone = backbone
#         self.head = head
#
#     def forward(self, graph):
#         features = self.backbone(graph)
#         return self.head(features, graph)
#
# model = MyModel(backbone, head)
# loss = CompositeLoss([
#     WeightedLoss(EnergyLoss(), weight=4.0, label="energy"),
#     WeightedLoss(ForcesLoss(), weight=100.0, label="forces"),
# ])
#
# trainer = MiniTrainer(
#     model=model,
#     loss_fn=loss,
#     optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
#     step_fn=graph_step,  # <-- Uses AtomicGraph batches
# )
# history = trainer.fit(graph_train_loader, graph_val_loader, epochs=50)

print("graph_step is available for AtomicGraph-based training.")
print("See the commented example above for usage.")

---

**Summary**: The `MiniTrainer` gives you a lightweight, notebook-friendly
training loop that operates entirely outside the Lightning/Hydra pipeline.
Extract features once from any foundation model, cache them, and iterate
on downstream heads rapidly.